# Live exercise: your data, your first number, your first run, worked

There is no single right answer here, because everybody's data is
different. So this is the exercise done once, on the worked example from
`sessions/session-11/solutions/worked-example-scope.md`, treating
`data/clean/plays.csv` as if it were "my own file". Use it to see what a
finished version of each step looks like, not as something to copy.

In [ ]:
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

## Part 1: read the file

In [ ]:
import pandas as pd

DATA_PATH = Path("data/clean/plays.csv")     # yours would be in project/data/

print(DATA_PATH, "exists:", DATA_PATH.exists())
assert DATA_PATH.exists()

### Three things that go wrong with files from a spreadsheet

In [ ]:
import io

dutch_csv = io.StringIO(
    "datum;categorie;bedrag\n"
    "2025-01-03;boodschappen;54,20\n"
    "2025-01-04;vervoer;12,85\n"
)

wrong = pd.read_csv(dutch_csv)
print(wrong.columns.tolist())
print(len(wrong.columns), "column")

assert len(wrong.columns) == 1

In [ ]:
dutch_csv.seek(0)
right = pd.read_csv(dutch_csv, sep=";", decimal=",")
print(right)
print(right.dtypes)

assert right["bedrag"].sum() == 67.05

Without `sep` there is one column and no error at all, which is the
dangerous part. With `sep=";"` and `decimal=","` there are three columns,
and `bedrag` is a real number (`float64`) that adds up to 67.05.

The general lesson: **a file that reads without an error has not
necessarily been read correctly.** The first look exists to catch exactly
this.

Our file is a plain comma-separated UTF-8 file, so it needs no extras.

In [ ]:
raw = pd.read_csv(DATA_PATH)

### The five-minute first look

In [ ]:
def first_look(df, date_column=None):
    """Print the four things to check on any new dataset."""
    print(f"rows: {len(df):,}   columns: {len(df.columns)}")
    print()
    print("types and missing values per column:")
    summary = pd.DataFrame({"type": df.dtypes, "missing": df.isna().sum()})
    print(summary)
    if date_column is not None:
        print()
        print(f"{date_column}: {df[date_column].min()} to {df[date_column].max()}")


first_look(raw, date_column="played_at")

**My notes, as they went into the worksheet:**

* 2,183 rows, 9 columns. The three I need are `played_at`, `genre` and
  `minutes_played`.
* `minutes_played` is `float64`, so it really is a number. `played_at` is
  text, which is fine: I only need the year, and that is the first four
  characters.
* nothing missing in any column.
* 2024-01-01 to 2025-12-31: two complete years, so a year-on-year
  comparison is fair. Nothing about the question needs to change.

On real data from a spreadsheet, expect at least one surprise here. That is
normal.

## Part 2: the one-number test

**My question:** which genres did I listen to more in 2025 than in 2024,
and which less?

**My must output:** a CSV with one row per genre: minutes in 2024, minutes
in 2025, and the change.

In [ ]:
raw["year"] = raw["played_at"].str[:4]

rough = raw.groupby(["genre", "year"])["minutes_played"].sum().unstack("year").round(1)
rough["change"] = (rough["2025"] - rough["2024"]).round(1)
rough = rough.sort_values("change")
rough

Five lines. The one-number test passes easily, so this project is not too
big. If anything it could carry one "could" item.

### Can I believe it?

The parts must add up to the whole, and one value checked by hand.

In [ ]:
assert abs(rough[["2024", "2025"]].sum().sum() - raw["minutes_played"].sum()) < 0.5

jazz_2025 = raw[(raw["genre"] == "Jazz") & (raw["year"] == "2025")]["minutes_played"].sum()
assert round(jazz_2025, 1) == rough.loc["Jazz", "2025"] == 533.2

print("both checks pass")

The second check works out one cell of the table a completely different
way, with a filter instead of a `groupby`, and gets the same 533.2. Two
routes to one number is the strongest cheap check there is. (Note the
brackets around each condition in the filter: the session 2 promise, kept.)

In [ ]:
gained = rough["change"].idxmax()
lost = rough["change"].idxmin()

print(f"From 2024 to 2025 I listened to {gained} for "
      f"{rough.loc[gained, 'change']:.1f} minutes more, and to {lost} for "
      f"{-rough.loc[lost, 'change']:.1f} minutes less.")

## Part 3: the first run of `project/pipeline.py`

This part happens in VS Code and the terminal. For the worked example it
went like this, with a run of `python project/pipeline.py` after every
step:

1. **Ran it unchanged.** It wrote `summary.csv` and `chart.png` to
   `project/output/`. Known-good starting point: yes.
2. **Pointed it at the file.** `INPUT = PROJECT.parent / "data" / "clean" /
   "plays.csv"`, because the worked example uses the course data. (Your
   own file lives in `project/data/`, so yours is
   `PROJECT / "data" / "..."`.) `QUESTION` became the question above. No
   extra `read_csv` arguments were needed.
3. **Cut `clean` down.** Kept `played_at`, `genre` and `minutes_played`,
   and added the `year` column. Nothing else.
4. **Replaced the body of `analyse`** with the five lines from part 2,
   ending in `.reset_index()` and `return` instead of displaying the
   table. Without `reset_index`, `save_csv` would have written the numbers
   without the genre names.
5. **Commented out the chart and the check** in `main()`, and the line
   each printed after it, with
   `# TODO (milestone 2): chart of the change per genre` and
   `# TODO (milestone 1): a check that fits this table`, because both
   expected the session 10 columns.
6. **Ran it**, and opened `project/output/summary.csv`: eight genres,
   Jazz `54.0` and Electronic `-106.1` in the change column, the same as
   above.

Then one commit, with only `project/pipeline.py` in it.

Notebook 01, step 7, shows what the four functions look like in this
shape, with an `assert` proving the function version gives exactly the
same table as the notebook version. That `assert` is worth copying into
your own work whenever code moves from a notebook into a script.